# Notebook 01 — Pretrain EEGNet on BCI Competition IV 2a

This notebook covers **Step 1** of the transfer learning pipeline:
- Load BCI IV 2a dataset (22 channels, 4 classes)
- Train EEGNet using 5-fold cross-validation to measure performance
- Retrain on 100% of data to produce a pretrained checkpoint for fine-tuning

> **Environment:** Google Colab T4 GPU

## 1. Environment Setup

In [ ]:
!cat /proc/cpuinfo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

: 

In [ ]:
import os
PROJECT_PATH = '/content/drive/MyDrive/Honours Project'
os.chdir(PROJECT_PATH)

In [ ]:
!pip install numpy torch braindecode scikit-learn pathlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from braindecode.models import EEGNet
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, cohen_kappa_score
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Use: {device}')

## 2. Load BCI IV 2a Dataset

BCI IV 2a is the standard benchmark dataset for Motor Imagery BCI research:
- **9 subjects**, 22 EEG channels, 250Hz sampling rate
- **4 classes**: Left Hand (0), Right Hand (1), Feet (2), Tongue (3)
- **288 trials/subject** → 5,184 trials total
- Pre-processed and saved as `.npz`

Data shape: `(n_trials, n_channels, n_timepoints)` = `(5184, 22, 1001)`

In [ ]:
# X shape: (5184, 22, 1001) → 5184 trials, 22 channels, 1001 timepoints
# y shape: (5184,)          → labels 0,1,2,3

d = np.load(os.path.join(PROJECT_PATH, 'bciv2a.npz'), allow_pickle=True)
X = d['X'].astype(np.float32)
y = d['y'].astype(np.int64)

print(f'X: {X.shape}')
print(f'y: {y.shape}, classes: {np.unique(y)}')

## 3. EEGNet Architecture

EEGNet is chosen for three reasons:
- **Parameter-efficient** (~2,500 params) — works well with limited EEG data
- **Two separable front-end layers** (temporal + spatial conv) — clean structure for transfer learning
- Proven effective across multiple BCI paradigms

```
Input (1, 22, 1001)
  → Temporal Conv  : learns frequency patterns (mu/beta rhythm)
  → Spatial Conv   : learns electrode weightings
  → Separable Conv : combines spatial-temporal features
  → Classifier     : 4-class output
```

In [ ]:
# Import EEGNetv4 from braindecode
# in_chans              → số channels EEG (22 cho BCI IV 2a)
# n_classes             → 4 class (left/right/feet/tongue)
# input_window_samples  → số timepoints (1001)

def make_model(n_chans, n_outputs, n_times):
    model = EEGNet(
        n_chans=n_chans,
        n_outputs=n_outputs,
        n_times=n_times,
        drop_prob=0.5
    )
    return model.to(device)

model = make_model(n_chans=22, n_outputs=4, n_times=1001).to(device)
print(model)

## 4. Training Utilities

Two helper functions kept separate for reusability:
- `train_epoch`: one full pass over training data, updates weights
- `evaluate`: computes **Accuracy** and **Cohen's Kappa** on test set

Cohen's Kappa measures improvement over random chance — more meaningful than raw accuracy when reporting in a thesis.

In [ ]:
# Duyệt qua từng batch, tính loss, cập nhật weights
# zero_grad → xóa gradient cũ trước mỗi batch
# backward  → tính gradient
# step      → cập nhật weights

def train_epoch(loader):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# Chạy model trên test set, không cập nhật weights
# argmax → lấy class có xác suất cao nhất

def evaluate(loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds   += model(X_batch.to(device)).argmax(dim=1).cpu().tolist()
            targets += y_batch.tolist()
    return accuracy_score(targets, preds), cohen_kappa_score(targets, preds)

## 5. 5-Fold Cross-Validation

5-fold CV splits data into 5 parts and rotates the test set across all folds so every trial is tested exactly once.

**Purpose: measure reliability, not produce a deployable model.** The 5 models trained here are discarded after evaluation.

- Low `std` → model is stable regardless of how data is split
- Results (mean ± std) are what gets reported in the thesis

In [ ]:
EPOCHS, BATCH = 500, 64
kfold   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

In [ ]:
# Mỗi fold: train 100 epoch → evaluate → ghi kết quả

for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y)):
    train_loader = DataLoader(TensorDataset(torch.tensor(X[train_idx]), torch.tensor(y[train_idx])), batch_size=BATCH, shuffle=True)
    test_loader  = DataLoader(TensorDataset(torch.tensor(X[test_idx]),  torch.tensor(y[test_idx])),  batch_size=BATCH)

    model     = make_model(n_chans=22, n_outputs=4, n_times=1001)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)

    for epoch in range(EPOCHS):
        loss = train_epoch(train_loader)
        scheduler.step(loss)

    acc, kappa = evaluate(test_loader)
    results.append((fold+1, acc, kappa))
    print(f'Fold {fold+1} → Acc: {acc:.4f} | Kappa: {kappa:.4f}')

In [ ]:
accs   = [r[1] for r in results]
kappas = [r[2] for r in results]
print(f'Trung bình → Acc: {np.mean(accs):.4f} ± {np.std(accs):.4f} | Kappa: {np.mean(kappas):.4f}')

## 6. Retrain on 100% Data → Save Pretrained Checkpoint

Once 5-fold CV confirms the model is reliable, retrain a single model on **all available data** to maximise the information encoded in the weights.

This checkpoint is used in **Notebook 02** to fine-tune onto MIMED (14 channels, Epoc X hardware).

> No validation set here — that was already handled by 5-fold CV above.

In [ ]:
# Train lại trên 100% data để save làm pretrained

full_loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)), batch_size=BATCH, shuffle=True)
model       = make_model(n_chans=22, n_outputs=4, n_times=1001).to(device)
optimizer   = torch.optim.Adam(model.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)

In [ ]:
for epoch in range(EPOCHS):
    loss = train_epoch(full_loader)
    scheduler.step(loss)
    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1} | Loss: {loss:.4f}')

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/Honours Project/eegnet_pretrained_500_new.pth')
print('Saved: eegnet_pretrained_500.pth')